[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [ ]:

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0

        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):

        B, S, _ = x.shape

        Q = self.W_q(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        new_len = Q.shape[2]
        d_k = K.shape[-1]

        if cache is not None:
          K = torch.cat([cache[0], K], dim = 2) #`stack` would create an outer dimension and requires operands to be same size, cat adds entry in a specific dimension increasing its #entries
          V = torch.cat([cache[1], V], dim = 2)

        tot_len = K.shape[2]
        past_len = tot_len - new_len

        raw_weights = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)

        k_idx = torch.arange(tot_len).unsqueeze(0) #(1, tot_len)
        q_idx = torch.arange(past_len, tot_len).unsqueeze(1) #(newlen, 1)

        mask = q_idx < k_idx #(new_len, tot_len)

        raw_weights = torch.masked_fill(raw_weights, mask, float("-inf"))

        att_weights = torch.softmax(raw_weights, dim = -1) #(B, num_heads, S, S)

        hidden = att_weights @ V #(B, num_heads, S, head_dim)
        hidden = hidden.transpose(1, 2).reshape((B, S, self.d_model)) #(B, S, num_heads * head_dim = d_model)

        return (self.W_o(hidden), (K, V))





In [13]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [14]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (6.8ms)
  ✅ [2/5] Cache structure (5.4ms)
  ✅ [3/5] Decode step appends to cache (3.7ms)
  ✅ [4/5] Incremental decode matches full forward (5.1ms)
  ✅ [5/5] Gradient flow (4.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (25.7ms total)
  Progress saved. Run status() to see your dashboard.

